<a href="https://colab.research.google.com/github/elhamod/BA305_Fall_2026/blob/main/Session%2009%20-%20Naive%20Bayes/lab5_Naive_Bayes_Spam_Filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Building a basic SPAM Filter with Multinomial Naive Bayes 
![picture](https://drive.google.com/uc?export=view&id=13fL79b9UydhOo-hCiWWNs2DkSP8pyu_P)

**dataset source:** University of California Irvine Spam DATABASE 'UCIspamdb.csv'
[link](https://archive.ics.uci.edu/ml/datasets/sms+spam+collection)

The collection is composed of just one text file, where each line has the correct class followed by the raw message. We offer some examples below:

ham or spam?  | email/sms text content
--------------|------------------
ham       | What you doing?how are you?
ham       |  Cos i was out shopping wif darren jus now n i called him 2 ask wat present he wan lor. Then he started guessing who i was wif n he finally guessed darren lor.
spam | Sunshine Quiz! Win a super Sony DVD recorder if you canname the capital of Australia? Text MQUIZ to 82277. B
spam | FreeMsg: Txt: CALL to No: 86888 & claim your reward of 3 hours talk time to use from your phone now! ubscribe6GBP/ mnth inc 3hrs 16 stop?txtStop

In [ ]:
# Import our usual packages...
import numpy as np
import pandas as pd

# Import some useful packages from scikit-learn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB # The Naive Bayes algo

In [ ]:
# Read in the dataset
df = pd.read_csv('https://raw.githubusercontent.com/AnalyticsArmory/data/main/UCIspamdb.csv')

In [ ]:
# Display the first 10 rows of the dataframe
df.head(10)

In [ ]:
# Display the info of the dataframe
df.info()

In [ ]:
# Replace the "spam"/"ham" tag with 1/0, using numpy's where() function.
df['target'] = np.where(df['target']=='spam',1, 0)

In [ ]:
# Display the first 10 rows of the dataframe
df.head(10)

In [ ]:
# Define inputs and target
X=df['text']
y=df['target']

In [ ]:
# Split the data: 70% for training, 30% for testing.
X_train, X_test, y_train, y_test = train_test_split(X,y , test_size=0.3, random_state=0)

## Vectorize Words

Because we are dealing with text, we need an intermediate step to **vectorize** the content, meaning turn it into digits. For example, imagine we have in our database entries:

$x_1$: `"Prescription drugs! Buy cheap prescription drugs drugs with big discount!"`

$x_2$: `"Your prescription drugs from CVS are ready for pickup."`

Step 1. We start by building a giant dictionary of words. Each row of text is converted into a vector of numbers, as follows:

text | and | ... | big | ...| buy | ... | cheap | ... | drugs | ... | pickup | ... | prescription | ... | is spam?
-----|----|-----|-----|-----|---|-----|-------|-----|-------|-----|---|---|---|---|---
$x_1$ | 1 | ... | 1 | ... | 1 | ... | 1 | ... | 3 | ... | 0 | ... | 2 | ...| yes
$x_2$  | 0 | ... | 0 | ... | 0 | ... | 0 | ... | 1 | ... | 1 | ... | 1 | ... | no

Step 2. The rest of the NB algorithm works as in the slides, from this point onwards. Let's assume we have a new entry: $x_3:$ "hello world". We vectorize $x_3$, then the algorithm can compute $p(x_3 \mid \text{spam})$ and then use Bayes' Theorem to compute $p(\text{spam} \mid x_3)$.


**`fit` vs. `transform`.** `fit()` builds the dictionary of every word in the **training** messages. `transform()` then counts how many times each dictionary word appears in each message. If a test message contains a word that never appeared in training, that word is simply skipped: the model cannot use a word it has never seen to make a prediction.

In [ ]:
# Build the dictionary from the training messages, then count words in both sets.
vectorizer = CountVectorizer().fit(X_train)
X_train_vec = vectorizer.transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(X_train_vec.shape)   # (number of messages, number of words in the dictionary)
print(X_test_vec.shape)

In [ ]:
# List the words the vectorizer found (its dictionary).
vectorizer.get_feature_names_out()

In [ ]:
# Show the word-count matrix: one row per message, one column per word.
print(X_train_vec.toarray())

The matrix is far too big to display, so all we see are zeros: most messages use only a handful of the ~7,000 dictionary words. To check that there are non-zero counts in there, we list every distinct number the matrix contains.

In [ ]:
# List every distinct count that appears anywhere in the matrix.
np.unique(X_train_vec.toarray())

## Drumroll.... Time to run the Naive Bayes Classifier

In [ ]:
# Train a multinomial Naive Bayes model on the word counts; we call it 'NB' for short.
NB = MultinomialNB()
NB.fit(X_train_vec, y_train)

In [ ]:
# How many training messages were ham (0) vs. spam (1)?
NB.class_count_

In [ ]:
# Accuracy of the NB model on the test set.
y_pred = NB.predict(X_test_vec)
accuracy_score(y_test, y_pred)

In [ ]:
# Confusion matrix of the NB model on the test set.
confusion_matrix(y_test, y_pred)

## Now, let's use our spam filter to classify some entirely new, never-before-seen text...

In [ ]:
# Four new messages the model has never seen.
emails=[
        "Hi all. How are you liking BA305 so far?",
        "Do you think this spam filter is working well?",
        "congratulations, you became today's lucky winner",
        "WINNER WINNER CHICKEN DINNER",
    ]

In [ ]:
# Vectorize the new messages and predict: 1 means spam, 0 means not spam.
emails_vec = vectorizer.transform(emails)
NB.predict(emails_vec)

**Question 1.** Look at the predictions above. Which of the four messages did the filter flag as spam, and do you agree with it?

**Answer**

*Provide your answer here*

It seems the keyword "winner" comes up often in spam. Let's try to trick the spam filter: can we write a perfectly normal message that uses "winner" and still gets it flagged? How good is our spam filter?

In [ ]:
# Four messages that use "winner" (or close to it) in a perfectly normal way.
emails2=[
        "congratulations, you became today's lucky winner",
        "Zendaya was the winner of our class project today",
        "Zendaya, I wanted to congratulate you as our winner today",
        "Zendaya, I wanted to congratulate you for winning the class project today",
    ]

In [ ]:
# Vectorize and predict: 1 means spam, 0 means not spam.
emails2_vec = vectorizer.transform(emails2)
NB.predict(emails2_vec)

**Question 2.** Which of the four messages above did the filter flag as spam? Did we manage to trick it?

**Answer**

*Provide your answer here*

## Bonus: How about we try to use our K-NN classifier from last time, to compare?

In [ ]:
# Train k-NN (k = 5) on the same word counts and score it on the test set.
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_vec, y_train)
y_pred_knn = knn.predict(X_test_vec)

print('Accuracy:', accuracy_score(y_test, y_pred_knn))
confusion_matrix(y_test, y_pred_knn)

In [ ]:
# How does k-NN classify the four "winner" messages?
knn.predict(emails2_vec)

**Question 3.** Compare k-NN with Naive Bayes on the test set: accuracy, and the confusion matrix's bottom row. Which model would you put in front of your inbox?

**Answer**

*Provide your answer here*